## Llamada a un endpoint falso

In [1]:
## Smoke-test: forzar un error de Binance (endpoint inválido)
from time import time
from panzer.rate_limit.binance_fixed import BinanceFixedWindowLimiter
from panzer.http.client import binance_public_get, BINANCE_SPOT_BASE_URL
from panzer.errors import BinanceAPIException

# Limiter muy permisivo (no queremos que bloquee nada aquí)
unsafe_limiter = BinanceFixedWindowLimiter(
    max_per_minute=10 ** 9,
    safety_ratio=1.0,
)

try:
    t0 = time()
    data, headers = binance_public_get(
        base_url=BINANCE_SPOT_BASE_URL,
        endpoint="/api/v3/this_does_not_exist",  # endpoint inventado
        params=None,
        limiter=unsafe_limiter,
        weight=1,
        timeout=5,
    )
    t1 = time()
    print(f"NO ERROR (raro). status OK, elapsed={t1 - t0:.3f}s, data={data}")
except BinanceAPIException as exc:
    print("BinanceAPIException capturada:")
    print("  status_code:", exc.status_code)
    if exc.error_payload:
        print("  code:", exc.error_payload.code)
        print("  msg:", exc.error_payload.msg)
except Exception as e:
    print("Excepción inesperada:", repr(e))


2025-11-27 21:13:19    ERROR [panzer.errors] BinanceAPIException raised: status=404 method=GET url=https://api.binance.com/api/v3/this_does_not_exist code=None msg=None


BinanceAPIException capturada:
  status_code: 404


## Inicializar cliente Panzer (SPOT) antes del stress test

In [2]:
from panzer.exchanges.binance.public import BinancePublicClient

client = BinancePublicClient()
print("Cliente BinancePublicClient(spot) inicializado.")

Cliente BinancePublicClient(spot) inicializado.


## FASE 1: Reventar /api/v3/exchangeInfo hasta obtener un 429 real

In [3]:
## FASE 1: Reventar /api/v3/exchangeInfo hasta obtener un 429 real
from time import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import json
import requests

BASE_URL = "https://api.binance.com"
ENDPOINT = "/api/v3/exchangeInfo"  # endpoint pesado

MAX_JOBS = 5000
MAX_WORKERS = 50

stop_event = threading.Event()


def parallel_worker(idx: int):
    if stop_event.is_set():
        return idx, None, None, None, None

    url = BASE_URL + ENDPOINT
    t0 = time()
    resp = requests.get(url, timeout=10)
    elapsed = time() - t0

    status = resp.status_code
    used_1m = resp.headers.get("x-mbx-used-weight-1m")

    if status != 200:
        stop_event.set()

    return idx, status, elapsed, used_1m, resp


print(f"Lanzando hasta {MAX_JOBS} jobs con {MAX_WORKERS} workers...")
t_start = time()

futures = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    for i in range(MAX_JOBS):
        if stop_event.is_set():
            break
        futures.append(pool.submit(parallel_worker, i))

print(f"Jobs lanzados: {len(futures)}. Esperando resultados...")

any_error = False
processed = 0

for fut in as_completed(futures):
    idx, status, elapsed, used_1m, resp = fut.result()
    if status is None:
        continue

    processed += 1
    print(f"[i={idx}] status={status} elapsed={elapsed:.3f}s used_weight_1m={used_1m}")

    if status != 200:
        any_error = True
        print("\n=== RESPUESTA NO 200 DETECTADA ===")
        print(f"Status: {status}")
        print("Headers:")
        print(resp.headers)

        try:
            body = resp.json()
        except json.JSONDecodeError:
            body = resp.text

        print("Body:")
        print(body)
        break

t_total = time() - t_start

if not any_error:
    print(
        f"\nFin del experimento sin respuestas != 200 "
        f"(processed={processed}, duración total={t_total:.1f}s)."
    )


Lanzando hasta 5000 jobs con 50 workers...
Jobs lanzados: 5000. Esperando resultados...
[i=20] status=200 elapsed=2.260s used_weight_1m=280
[i=21] status=200 elapsed=1.872s used_weight_1m=440
[i=19] status=200 elapsed=0.712s used_weight_1m=100
[i=18] status=200 elapsed=1.258s used_weight_1m=200
[i=16] status=200 elapsed=2.149s used_weight_1m=540
[i=17] status=200 elapsed=2.206s used_weight_1m=820
[i=15] status=200 elapsed=2.145s used_weight_1m=500
[i=14] status=200 elapsed=1.941s used_weight_1m=460
[i=13] status=200 elapsed=1.942s used_weight_1m=240
[i=315] status=429 elapsed=0.360s used_weight_1m=6320

=== RESPUESTA NO 200 DETECTADA ===
Status: 429
Headers:
{'Content-Type': 'application/json;charset=UTF-8', 'Content-Length': '175', 'Connection': 'keep-alive', 'Date': 'Thu, 27 Nov 2025 20:13:36 GMT', 'Expires': '0', 'Server': 'nginx', 'x-mbx-uuid': '6b52b59c-b6e9-452e-b3ce-e1d3de5d21ce', 'retry-after': '24', 'x-mbx-used-weight': '6320', 'x-mbx-used-weight-1m': '6320', 'Strict-Transport

## FASE 2: Probar Panzer después de haber consumido el rate limit

In [5]:
## FASE 2: Probar el limitador de Panzer tras el abuso
from time import time
from datetime import datetime
import requests

from panzer.exchanges.binance.public import BinancePublicClient

BASE_URL = "https://api.binance.com"
TIME_ENDPOINT = "/api/v3/time"

# Cliente Panzer para esta fase de pruebas

# 1) Estado crudo del peso justo después del stress test
t0 = time()
resp = requests.get(BASE_URL + TIME_ENDPOINT, timeout=5)
elapsed_raw = time() - t0
used_1m_raw = resp.headers.get("x-mbx-used-weight-1m")

print("== Estado justo después del stress test ==")
print(
    f"status={resp.status_code} elapsed={elapsed_raw:.3f}s "
    f"used_weight_1m={used_1m_raw}"
)

print("\nLanzando varias llamadas server_time() vía Panzer...")
N = 5

for i in range(N):
    t0 = time()
    exc = None

    try:
        server_time = client.server_time(timeout=5)
    except Exception as e:
        server_time = None
        exc = e

    t1 = time()
    elapsed = t1 - t0

    used_local = client.limiter.used_local
    server_used = client.limiter.last_server_used

    print(
        f"[{i}] {datetime.utcnow().isoformat(timespec='seconds')} "
        f"elapsed={elapsed:.3f}s | used_local={used_local} | "
        f"server_used={server_used} | exc={repr(exc)}"
    )


== Estado justo después del stress test ==
status=429 elapsed=0.293s used_weight_1m=6322

Lanzando varias llamadas server_time() vía Panzer...


AssertionError: Limiter no inicializado

In [ ]:
print("Terminado")